# Mushroom Model Hyperparameter Tuning

Find hyperparameters for the four base classifiers using training-only cross-validation, compare them on validation data, and save the selected settings for `3-Models-Final.ipynb`.


## Imports

Load the machine-learning, data, plotting, and pickle tools needed for the search.


In [4]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

## Load, clean, and split data

Use the same deterministic stratified 70% training / 20% validation / 10% test split as the final notebook. Search only on training data; validation compares the selected settings, while test data is kept untouched.


In [5]:
# Load the CSV saved in the parent mushrooms folder.
dataset_path = Path('../mushroom_project_dataset.csv')
df = pd.read_csv(dataset_path)

# Remove noise / sparse columns identified during data analysis; ignore missing names safely.
df = df.drop(columns=['jumbled_noise_0', 'jumbled_noise_1', 'spore-print-color'], errors='ignore')
if 'class' not in df.columns:
    raise ValueError("Dataset must contain a 'class' target column.")

X = df.drop(columns='class')
y = df['class'].astype(str)
class_labels = sorted(y.unique())
if len(class_labels) != 2:
    raise ValueError(f'Expected a binary mushroom target, found classes: {class_labels}')

# Stratified 70/20/10 split: reserve test first, then split the other 90%.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=2/9, random_state=42, stratify=y_train_val
)
print(f'Train: {len(X_train)} rows ({len(X_train) / len(X):.0%})')
print(f'Validation: {len(X_val)} rows ({len(X_val) / len(X):.0%})')
print(f'Test: {len(X_test)} rows ({len(X_test) / len(X):.0%})')
print('Class labels:', class_labels)

# Fit imputation and encoding only inside training folds via model pipelines.
numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_preprocessing, numeric_columns),
    ('categorical', categorical_preprocessing, categorical_columns),
])

Train: 3500 rows (70%)
Validation: 1000 rows (20%)
Test: 500 rows (10%)
Class labels: ['e', 'p']


## Search for the best parameters

Search eight randomized combinations per model with three-fold stratified cross-validation, optimizing accuracy. Preprocessing remains inside each pipeline and is fitted separately within each fold. The best parameter dictionaries and validation/CV statistics are saved to `models/best_model_parameters.pkl`. Only load pickle files you created or trust.


In [6]:
# XGBoost requires numeric classes; retain the mapping for readable reports.
label_map = {label: index for index, label in enumerate(class_labels)}
y_train_encoded = np.array([label_map[label] for label in y_train])
y_val_encoded = np.array([label_map[label] for label in y_val])
encoded_labels = [label_map[label] for label in class_labels]

base_estimators = {
    'decision_tree': DecisionTreeClassifier(random_state=42),
    'random_forest': RandomForestClassifier(n_jobs=-1, random_state=42),
    'gradient_boosting': GradientBoostingClassifier(random_state=42),
    'xgboost': XGBClassifier(eval_metric='logloss', n_jobs=1, random_state=42),
}

parameter_distributions = {
    'decision_tree': {
        'classifier__max_depth': [None, 5, 10, 15, 20],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__min_samples_leaf': [1, 2, 4],
        'classifier__class_weight': [None, 'balanced'],
    },
    'random_forest': {
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__min_samples_leaf': [1, 2, 4],
        'classifier__max_features': ['sqrt', 'log2'],
        'classifier__class_weight': [None, 'balanced', 'balanced_subsample'],
    },
    'gradient_boosting': {
        'classifier__n_estimators': [50, 100, 150, 200],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__max_depth': [1, 2, 3],
        'classifier__subsample': [0.7, 0.85, 1.0],
    },
    'xgboost': {
        'classifier__n_estimators': [50, 100, 150, 200],
        'classifier__max_depth': [3, 4, 6],
        'classifier__learning_rate': [0.03, 0.05, 0.1],
        'classifier__subsample': [0.7, 0.85, 1.0],
        'classifier__colsample_bytree': [0.7, 0.85, 1.0],
    },
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
searches = {}
best_params = {}
best_cv_scores = {}
validation_metrics = {}

for name, estimator in base_estimators.items():
    print(f'\nSearching {name.replace("_", " ")} ...')
    model_pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', estimator),
    ])
    search = RandomizedSearchCV(
        estimator=model_pipeline,
        param_distributions=parameter_distributions[name],
        n_iter=8,
        scoring='accuracy',
        cv=cv,
        n_jobs=1,
        random_state=42,
        refit=True,
        error_score='raise',
    )
    search.fit(X_train, y_train_encoded)
    searches[name] = search
    best_params[name] = search.best_params_
    best_cv_scores[name] = float(search.best_score_)
    val_predictions = search.best_estimator_.predict(X_val)
    validation_metrics[name] = {
        'accuracy': accuracy_score(y_val_encoded, val_predictions),
        'classification_report': classification_report(
            y_val_encoded, val_predictions, labels=encoded_labels,
            target_names=class_labels, output_dict=True, zero_division=0,
        ),
        'confusion_matrix': confusion_matrix(y_val_encoded, val_predictions, labels=encoded_labels),
    }
    print(f'Best CV accuracy: {search.best_score_:.3f}')
    print('Best parameters:', search.best_params_)

search_summary = pd.DataFrame({
    name: {
        'best_cv_accuracy': best_cv_scores[name],
        'validation_accuracy': validation_metrics[name]['accuracy'],
        'validation_edible_precision': validation_metrics[name]['classification_report']['e']['precision'],
        'validation_edible_recall': validation_metrics[name]['classification_report']['e']['recall'],
        'validation_poisonous_recall': validation_metrics[name]['classification_report']['p']['recall'],
    }
    for name in base_estimators
}).T.sort_values('best_cv_accuracy', ascending=False)
print('\nSearch and validation summary (accuracy is the search objective):')
print(search_summary.to_string(float_format=lambda value: f'{value:.3f}'))

output_dir = Path('models')
output_dir.mkdir(parents=True, exist_ok=True)
parameter_artifact = {
    'format_version': 1,
    'best_params': best_params,
    'best_cv_scores': best_cv_scores,
    'validation_metrics': validation_metrics,
    'search_scoring': 'accuracy',
    'cv_splits': 3,
    'n_iter': 8,
    'class_labels': class_labels,
    'label_map': label_map,
    'feature_columns': X.columns.tolist(),
}
parameter_path = output_dir / 'best_model_parameters.pkl'
with parameter_path.open('wb') as parameter_file:
    pickle.dump(parameter_artifact, parameter_file, protocol=pickle.HIGHEST_PROTOCOL)
print(f'\nSaved selected parameters to: {parameter_path.resolve()}')



Searching decision tree ...
Best CV accuracy: 0.712
Best parameters: {'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 2, 'classifier__max_depth': 20, 'classifier__class_weight': None}

Searching random forest ...
Best CV accuracy: 0.768
Best parameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 1, 'classifier__max_features': 'log2', 'classifier__max_depth': None, 'classifier__class_weight': 'balanced'}

Searching gradient boosting ...
Best CV accuracy: 0.726
Best parameters: {'classifier__subsample': 0.85, 'classifier__n_estimators': 150, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.05}

Searching xgboost ...
Best CV accuracy: 0.741
Best parameters: {'classifier__subsample': 0.85, 'classifier__n_estimators': 100, 'classifier__max_depth': 4, 'classifier__learning_rate': 0.1, 'classifier__colsample_bytree': 0.85}

Search and validation summary (accuracy is the search objective):
               